# About

What we're trying to do is use master_units.parquet as our new source of truth, while also adding in things that we feel like are necessary in order to get what we need out of our apartments scrape. The first part is getting the right subset of apartments using the addresses, and then the second part is just doing a cron job with checkpoints to scrape all of them. 

In [1]:
import pandas as pd
import os
os.chdir('/home/kevinphan981/Programming/apartment-rental-scraper/')
df = pd.read_parquet('data/master_units.parquet', engine = 'pyarrow')
master_address = pd.read_csv('data/master_tmk_address.csv')
print(df.head(n = 10))
print(master_address.head(n = 10))
print(df.shape) # about 6000, which is around what we're looking for.

     master_tmk parcel_number unit_number      modified_address
0  210020160000  210020160001        1401  66 QUEEN ST APT 1401
1  210020160000  210020160002        1402  66 QUEEN ST APT 1402
2  210020160000  210020160003        1405  66 QUEEN ST APT 1405
3  210020160000  210020160004        1501  66 QUEEN ST APT 1501
4  210020160000  210020160005        1502  66 QUEEN ST APT 1502
5  210020160000  210020160006        1503  66 QUEEN ST APT 1503
6  210020160000  210020160007        1504  66 QUEEN ST APT 1504
7  210020160000  210020160008        1505  66 QUEEN ST APT 1505
8  210020160000  210020160009        1601  66 QUEEN ST APT 1601
9  210020160000  210020160010        1602  66 QUEEN ST APT 1602
   Unnamed: 0         parid       tmk  suffix  seq  taxyr unitno  totacres  \
0           0  230220520294  23022052     294    2   2027   2006       0.0   
1           2  260140320188  26014032     188    0   2027    904       0.0   
2           3  260140320191  26014032     191    0   2027    9

In [2]:
# let's try to join master_tmk_address.csv valid addresses (which we can only guess is right) with the master_units.parquet (which is definitely right), with the condo_listings, which is definitely not right
df['master_tmk'] = df['master_tmk'].astype('int64')
df['parcel_number'] = df['parcel_number'].astype('int64')

df_1 = df.merge(master_address, left_on= 'parcel_number', right_on = 'parid', how = 'left')

print(df_1.shape)

(5824, 25)


In [ ]:
# we now have to iterate on the other things and then do a close approximation of the data
# for some reason there is an import error for qpublic_unit but it works regardless

os.chdir('/home/kevinphan981/Programming/apartment-rental-scraper/src/qpublic')
os.getcwd()
os.listdir() # does contain qpublic_unit.py
from qpublic_unit import fetch_and_parse_parcel, json_to_dataframe

example_parcel = "260200690004"

record = fetch_and_parse_parcel(example_parcel)

record_df = json_to_dataframe(record)
print(record_df.head())


In [ ]:
# the issue here is that I have to go through every single one, which is not efficient
# I would rather use the subset of units that I have, roughly join them by address, and then use the parcel #s from master_tmk_address.csv to do it

final_df = pd.DataFrame()

for parcel in df['parcel_number'][:3]:
    record = fetch_and_parse_parcel(parcel)
    record_df = json_to_dataframe(record)
    pd.concat([final_df, record_df], ignore_index=True)
    print(f"Completed {parcel}")

print(final_df.head())